In [1]:
import hashlib
import time

class Block:
    def __init__(self, index, previous_hash, transactions):
        self.index = index
        self.timestamp = time.time()
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.nonce = 0
        self.hash = self.compute_hash()

    def compute_hash(self):
        block_string = f"{self.index}{self.timestamp}{self.transactions}{self.previous_hash}{self.nonce}"
        return hashlib.sha256(block_string.encode()).hexdigest()

    def mine_block(self, difficulty):
        while self.hash[:difficulty] != '0' * difficulty:
            self.nonce += 1
            self.hash = self.compute_hash()

class Blockchain:
    def __init__(self):
        self.chain = []
        self.difficulty = 2
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis_block = Block(0, "0", [])
        self.chain.append(genesis_block)

    def add_block(self, transactions):
        previous_block = self.chain[-1]
        new_block = Block(len(self.chain), previous_block.hash, transactions)
        new_block.mine_block(self.difficulty)
        self.chain.append(new_block)

    def mine(self):
        self.add_block([])
        return len(self.chain) - 1


In [2]:
class BankAccount:
    def __init__(self, account_id, name, balance=0):
        self.account_id = account_id
        self.name = name
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if self.balance >= amount:
            self.balance -= amount
            return True
        return False

    def to_dict(self):
        return {
            "account_id": self.account_id,
            "name": self.name,
            "balance": self.balance
        }


In [3]:
def transfer_asset(sender, receiver, amount, accounts):
    if sender not in accounts or receiver not in accounts:
        raise Exception("Invalid account(s)")

    accounts[sender].debit(amount)
    accounts[receiver].credit(amount)

    return {
        "type": "transfer",
        "from": sender,
        "to": receiver,
        "amount": amount,
        "timestamp": time.time()
    }


In [4]:
import time

def create_transaction(sender_id, receiver_id, amount, accounts, blockchain):
    if sender_id not in accounts or receiver_id not in accounts:
        return {"error": "Invalid account ID"}

    sender = accounts[sender_id]
    receiver = accounts[receiver_id]

    if not sender.withdraw(amount):
        return {"error": "Insufficient funds"}

    receiver.deposit(amount)

    txn = {
        "sender": sender_id,
        "receiver": receiver_id,
        "amount": amount,
        "timestamp": time.time()
    }

    blockchain.add_block([txn])
    return {"status": "Transaction successful", "transaction": txn}


In [5]:
from blockchain import Blockchain
from bank_asset_manager import BankAccount
from transactions import create_transaction

accounts = {
    "1": BankAccount("1", "Abdullah", 1000),
    "2": BankAccount("2", "Asif", 500)
}

blockchain = Blockchain()

def main():
    while True:
        print("\n1. View Accounts\n2. Transfer Asset\n3. Mine Block\n4. View Blockchain\n5. Exit")
        choice = input("Enter choice: ")

        if choice == '1':
            for acc in accounts.values():
                print(acc.to_dict())

        elif choice == '2':
            s = input("Sender ID: ")
            r = input("Receiver ID: ")
            try:
                amt = float(input("Amount: "))
            except ValueError:
                print("Invalid amount. Must be a number.")
                continue

            result = create_transaction(s, r, amt, accounts, blockchain)
            print("Transaction Result:", result)

        elif choice == '3':
            index = blockchain.mine()
            print(f"Block {index} mined.")

        elif choice == '4':
            for block in blockchain.chain:
                print(vars(block))

        elif choice == '5':
            break
        else:
            print("Invalid option.")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'blockchain'